# FedLabel on OrganAMNIST: reproducing the ICCV 2023 paper

**Setup from the paper (ICCV 2023 + arXiv appendix) and our ADD:**
- 10 clients, Dirichlet alpha (0.1 and 1.0), 30% participation (3 clients per round), ResNet18, 800 rounds
- 5% labeled (Table 3 / 4) or 20% labeled (Fig. 4b); labeled/unlabeled split uniformly at random per client
- RandAugment(1, 10), variance confidence, lambda_0 = 1.0, 3 seeds, mean (std)
- beta = 0.5: not stated in the paper; the appendix ablation (Fig. 8) is best around 0.3 to 0.5
- not stated in the paper, kept fixed: lr 0.01, batch 20, tau = tau' = 15, SGD momentum 0.9, weight decay 5e-4

**Data split.** Main experiments use MedMNIST's official, patient-level train/test split (`split='official'`,
test = 17,778 images), as our ADD baselines did. The appendix's pooled 80/5/15 split (`split='paper'`) mixes
the CT slices of the same patient into train and test; there FedAvg with 5% labels reached 81%, above the
paper's 71% for fully labeled FedAvg, so it is kept only as a reference section.

**Metrics (ADD).** Top-1 accuracy on the global test set (primary), macro precision / recall / F1 (printed at
the end of every run), and the relative ratio accuracy / FedAvg (100%) for comparing with the paper.

Run cells 1 and 2 first; after that every experiment cell can be run on its own. Logs go to Drive; a run that
already finished is skipped, so after a Colab disconnect just rerun cells 1, 2 and the interrupted cell.

In [5]:
# 1) Drive, repo (branch test1), requirements
import os
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)

REPO_URL = 'https://github.com/ozanyildirim22/federated-learning.git'
REPO_DIR = '/content/federated-learning'
BRANCH   = 'test1'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git fetch -q origin
!git checkout -q {BRANCH} 2>/dev/null || git checkout -q -b {BRANCH} origin/{BRANCH}
!git reset -q --hard origin/{BRANCH}
!git log -1 --oneline
!pip install -q fedlab medmnist path

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (Runtime > Change runtime type > GPU)')

/content/federated-learning
2f0ecc2 (HEAD -> test1, origin/test1) Use patient-level OrganAMNIST split as main protocol, add macro F1
GPU: Tesla T4


In [6]:
# 2) Helpers: data cache on Drive, experiment runner, result table
import os, re, glob, shutil, statistics
from IPython import get_ipython

DRIVE_ROOT = '/content/drive/MyDrive/federated-learning-data'
LOG_DIR    = os.path.join(DRIVE_ROOT, 'logs', 'organamnist_paper')
os.makedirs(LOG_DIR, exist_ok=True)

# Paper settings + the values the paper does not state (lr, batch, tau, tau'; beta from the appendix ablation)
COMMON = dict(rounds=800, clients_per_round=3, batch_size=20, lr=0.01,
              beta=0.5, lambda_0=1.0, tau=15, tau_prime=15)


def data_name(split, alpha, label_ratio, seed):
    if split == 'paper':
        return f'pickles_paper_alpha{alpha}_label{label_ratio}_seed{seed}'
    return f'pickles_alpha{alpha}_label{label_ratio}' + (f'_seed{seed}' if seed else '')


def prepare_data(alpha, label_ratio, seed, split='official'):
    """Client pickles for one (split, alpha, label_ratio, seed): local -> Drive zip -> preprocess.
    split='official': MedMNIST patient-level train/test, test = 17,778 images (main experiments)
    split='paper'   : pooled 80/5/15 (FedLabel appendix), test = 8,825 images (leaks slices, reference only)"""
    name       = data_name(split, alpha, label_ratio, seed)
    local_base = os.path.join(REPO_DIR, 'data', 'organamnist')
    local_dir  = os.path.join(local_base, name)
    drive_dir  = os.path.join(DRIVE_ROOT, 'organamnist')
    # official zips get an 'official_' prefix so an older cache with the same folder name is never reused
    drive_zip  = os.path.join(drive_dir, ('' if split == 'paper' else 'official_') + f'{name}.zip')
    if os.path.isdir(local_dir):
        print(f'[local] {name}')
    elif os.path.isfile(drive_zip):
        shutil.unpack_archive(drive_zip, local_base)
        print(f'[drive] extracted {os.path.basename(drive_zip)}')
    else:
        get_ipython().system(
            f'cd {REPO_DIR} && PYTHONPATH=. python data/organamnist/preprocess.py '
            f'--split {split} --alpha {alpha} --label_ratio {label_ratio} --seed {seed}')
        if not os.path.isdir(local_dir):
            raise RuntimeError(f'{local_dir} was not created, check the output above')
        os.makedirs(drive_dir, exist_ok=True)
        tmp = shutil.make_archive(f'/content/_{name}', 'zip', root_dir=local_base, base_dir=name)
        shutil.copy(tmp, drive_zip); os.remove(tmp)
        print(f'[drive] cached {os.path.basename(drive_zip)}')


def run(algorithm, alpha, label_ratio, seed, split='official', force=False, **over):
    """algorithm: fedlabel | fedavg | fedavg_full; split: official | paper. The log goes to Drive.
    A run whose log on Drive already has a final result is skipped (force=True reruns it), so a loop
    cell can simply be rerun after a Colab disconnect and continues where it stopped."""
    p = {**COMMON, **over}
    tag = f'{algorithm}{"" if split == "paper" else "_official"}_a{alpha}_l{label_ratio}_b{p["beta"]}_s{seed}'
    log = os.path.join(LOG_DIR, f'{tag}.txt')
    if not force and os.path.isfile(log) and re.search(r'Global .+ accuracy: [\d.]+%', open(log, errors='ignore').read()):
        print(f'[skip] {tag} already finished (force=True to rerun)')
        return
    prepare_data(alpha, label_ratio, seed, split)
    get_ipython().system(
        f'cd {REPO_DIR} && PYTHONPATH=. python fedlabel/main.py '
        f'--algorithm {algorithm} --dataset organamnist --model resnet18 '
        f'--split {split} --seed {seed} --alpha {alpha} --label_ratio {label_ratio} '
        f'--comms_round {p["rounds"]} --client_num_per_round {p["clients_per_round"]} '
        f'--batch_size {p["batch_size"]} --local_lr {p["lr"]} '
        f'--beta {p["beta"]} --lambda_0 {p["lambda_0"]} --tau {p["tau"]} --tau_prime {p["tau_prime"]} '
        f'--confidence_metric variance --global_test True --eval_every 50 '
        f'2>&1 | tee {log}')


# Paper numbers (OrganAMNIST column): (alpha, label_ratio, algorithm) -> (mean, std)
PAPER = {
    (0.1, 0.05, 'fedlabel'):    (69.43, 1.58),   # Table 3
    (0.1, 0.05, 'fedavg'):      (41.90, 2.18),
    (0.1, 0.05, 'fedavg_full'): (71.09, 1.00),
    (1.0, 0.05, 'fedlabel'):    (79.76, 0.83),   # Table 4
    (1.0, 0.05, 'fedavg'):      (59.74, 1.52),
    (1.0, 0.05, 'fedavg_full'): (81.49, 0.96),
}


def summary():
    """acc   = accuracy after the last round (mean (std) over seeds)
    last4 = mean accuracy of the last 4 evaluations (rounds 650-800), less sensitive to
            which 3 clients happened to train in the final round
    f1    = macro F1 after the last round (ADD: secondary metric for imbalanced OrganAMNIST)
    ratio = acc / acc of FedAvg (100%) with the same split and alpha (ADD's relative accuracy
            ratio); the paper's own ratio is shown next to it"""
    ansi = re.compile(r'\x1b\[[0-9;]*m')
    rows = {}
    for f in sorted(glob.glob(os.path.join(LOG_DIR, '*.txt'))):
        m = re.match(r'(fedlabel|fedavg_full|fedavg)(_official)?_a([\d.]+)_l([\d.]+)_b([\d.]+)_s(\d+)\.txt', os.path.basename(f))
        if not m:
            continue
        text = ansi.sub('', open(f, errors='ignore').read())
        final = re.findall(r'Global .+ accuracy: ([\d.]+)%', text)
        if not final:
            continue  # run not finished
        f1 = re.findall(r'Global .+ macro-F1: ([\d.]+)%', text)
        evals = [float(x) for x in re.findall(r'\[round \d+\] .*?acc ([\d.]+)%', text)]
        last4 = statistics.mean(evals[-4:]) if evals else float(final[-1])
        alg, off, a, l, b, s = m.groups()
        split = 'official' if off else 'paper'
        rows.setdefault((split, float(a), float(l), alg, float(b)), []).append(
            (float(final[-1]), last4, float(f1[-1]) if f1 else None))

    def ms(xs):
        xs = [x for x in xs if x is not None]
        if not xs:
            return '-'
        sd = statistics.pstdev(xs) if len(xs) > 1 else 0.0
        return f'{statistics.mean(xs):.2f} ({sd:.2f})'

    full = {}  # (split, alpha) -> final accuracies of FedAvg (100%)
    for (split, a, l, alg, b), res in rows.items():
        if alg == 'fedavg_full':
            full.setdefault((split, a), []).extend(r[0] for r in res)

    print(f'{"split":<8} {"alpha":>5} {"label":>5} {"algorithm":<12} {"beta":>4} {"seeds":>5} '
          f'{"acc":>14} {"last4":>14} {"f1":>14} {"ratio":>6} | {"paper acc":>14} {"ratio":>6}')
    for (split, a, l, alg, b), res in sorted(rows.items()):
        acc_mean = statistics.mean(r[0] for r in res)
        ref = full.get((split, a))
        ratio = f'{acc_mean / statistics.mean(ref):.3f}' if ref else '-'
        pm, pf = PAPER.get((a, l, alg)), PAPER.get((a, l, 'fedavg_full'))
        paper = f'{pm[0]:.2f} ({pm[1]:.2f})' if pm else '-'
        pratio = f'{pm[0] / pf[0]:.3f}' if pm and pf else '-'
        print(f'{split:<8} {a:>5} {l:>5} {alg:<12} {b:>4} {len(res):>5} '
              f'{ms([r[0] for r in res]):>14} {ms([r[1] for r in res]):>14} {ms([r[2] for r in res]):>14} '
              f'{ratio:>6} | {paper:>14} {pratio:>6}')

## A. Table 3 setting: alpha = 0.1, 5% labeled (main result)
Paper: FedLabel 69.43 (1.58), FedAvg 41.90 (2.18), FedAvg (100%) 71.09 (1.00). Our ADD FedAvg baselines: 49.25 (5%), 59.16 (100%).
Run seed 0 of all three first and check `summary()`, then seeds 1 and 2.

In [ ]:
run('fedlabel', alpha=0.1, label_ratio=0.05, seed=0)

/usr/local/lib/python3.13/dist-packages/torch/cuda/__init__.py:68: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
19:05:04 - numexpr.utils - INFO - NumExpr defaulting to 2 threads.
client 0: labeled 155 / unlabeled 2963 / test 1785
client 1: labeled 145 / unlabeled 2765 / test 2723
client 2: labeled 185 / unlabeled 3534 / test 1877
client 3: labeled 170 / unlabeled 3249 / test 1793
client 4: labeled 192 / unlabeled 3665 / test 1446
client 5: labeled 311 / unlabeled 5913 / test 1204
client 6: labeled 92 / unlabeled 1763 / test 1827
client 7: labeled 212 / unlabeled 4038 / test 626
client 8: labeled 15 / unlabeled 289 / test 2286
client 9: labeled 245 / unlabeled 4660 / test 2211
[drive] cached official_pickles_alpha0.1_label0.05.zip
/usr/local/lib/python3.13/dist-packages/torch/cuda/__in

In [ ]:
run('fedavg', alpha=0.1, label_ratio=0.05, seed=0)

In [ ]:
run('fedavg_full', alpha=0.1, label_ratio=0.05, seed=0)

In [ ]:
for seed in (1, 2):
    for alg in ('fedlabel', 'fedavg', 'fedavg_full'):
        run(alg, alpha=0.1, label_ratio=0.05, seed=seed)

## B. Table 4 setting: alpha = 1.0, 5% labeled (low heterogeneity)
Paper: FedLabel 79.76 (0.83), FedAvg 59.74 (1.52), FedAvg (100%) 81.49 (0.96). Our ADD FedAvg baseline: 79.46 (5%).

In [ ]:
for seed in (0, 1, 2):
    for alg in ('fedlabel', 'fedavg', 'fedavg_full'):
        run(alg, alpha=1.0, label_ratio=0.05, seed=seed)

## C. Fig. 4b setting: alpha = 0.1, 20% labeled
The paper shows this only as a curve: FedLabel beats the baselines by 2 to 10 points. FedAvg (100%) does not depend on the
label ratio, so the ratio column reuses the one from section A.

In [ ]:
for seed in (0, 1, 2):
    for alg in ('fedlabel', 'fedavg'):
        run(alg, alpha=0.1, label_ratio=0.2, seed=seed)

## D. Optional (ADD): alpha = 1.0, 20% labeled
Not in the paper's tables; our ADD lists both alphas for 20% (FedAvg baseline 85.36).

In [ ]:
for seed in (0, 1, 2):
    for alg in ('fedlabel', 'fedavg'):
        run(alg, alpha=1.0, label_ratio=0.2, seed=seed)

## E. Optional: beta check
The paper does not state the beta used in its tables. Same setting as A, seed 0, with the high and a lower threshold.

In [ ]:
run('fedlabel', alpha=0.1, label_ratio=0.05, seed=0, beta=0.95)
run('fedlabel', alpha=0.1, label_ratio=0.05, seed=0, beta=0.4)

## F. Reference: pooled 80/5/15 split (arXiv appendix)
Shows how much the slice leakage inflates the numbers. Runs that already finished are skipped and show up in `summary()` as `paper`.

In [ ]:
for alg in ('fedlabel', 'fedavg', 'fedavg_full'):
    run(alg, alpha=0.1, label_ratio=0.05, seed=0, split='paper')

## Results vs paper

In [ ]:
summary()